# Analyse de Prédictions de Victoire Pokemon

## Objectifs
- **Nettoyage des données** : Gérer les valeurs manquantes et préparer les données pour l'analyse
- **Exploration des données** : Visualiser les corrélations et les tendances principales
- **Engineering des features** : Créer de nouvelles caractéristiques pertinentes
- **Modélisation** : Construire et comparer plusieurs modèles de régression (Linéaire, SVM, Decision Trees)
- **Dimensionnalité réduite** : Appliquer PCA pour la réduction dimensionnelle

## Instructions
Cet exercice vous demande d'analyser les données Pokemon pour prédire le taux de victoire. Vous devez :
1. Charger et fusionner les fichiers pokemon.csv et combats.csv
2. Nettoyer les données (gérer les valeurs manquantes)
3. Effectuer une exploration complète avec visualisations
4. Entraîner 3 modèles de régression différents
5. Comparer et évaluer les performances avec MAE
6. Appliquer PCA pour la réduction dimensionnelle

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set visualization parameters
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# 1. LOAD AND MERGE DATA
print('=== LOADING DATA ===')
pokemon_url = 'https://raw.githubusercontent.com/KeithGalli/pokemon/master/pokemon.csv'
combats_url = 'https://raw.githubusercontent.com/KeithGalli/pokemon/master/combats.csv'

pokemon_df = pd.read_csv(pokemon_url)
combats_df = pd.read_csv(combats_url)

print(f'Pokemon dataset shape: {pokemon_df.shape}')
print(f'Combats dataset shape: {combats_df.shape}')
print(f'\nPokemon columns: {pokemon_df.columns.tolist()}')
print(f'Combats columns: {combats_df.columns.tolist()}')

# 2. DATA CLEANING
print('\n=== DATA CLEANING ===')
# Handle missing values in Pokemon dataset
print(f'Missing values before cleaning:')
print(pokemon_df.isnull().sum())

# Fill missing Type 2 with 'None'
pokemon_df['Type 2'].fillna('None', inplace=True)

# Fix Pokemon #62 (Poliwrath) missing name
if pokemon_df.loc[pokemon_df['#'] == 62, 'Name'].isnull().any():
    pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = 'Poliwrath'

print(f'\nMissing values after cleaning: {pokemon_df.isnull().sum().sum()}')

# 3. CALCULATE WIN PERCENTAGE
print('\n=== CALCULATING WIN PERCENTAGE ===')
win_counts = combats_df['Winner'].value_counts().reset_index()
win_counts.columns = ['#', 'Wins']

total_battles = pd.concat([combats_df['Winner'], combats_df['Loser']]).value_counts().reset_index()
total_battles.columns = ['#', 'Total_Battles']

win_percentage = win_counts.merge(total_battles, on='#', how='left')
win_percentage['Win_Rate'] = (win_percentage['Wins'] / win_percentage['Total_Battles'] * 100).round(2)

print(f'Pokemon with battles: {len(win_percentage)}')
print(f'Win Rate statistics:')
print(win_percentage['Win_Rate'].describe())

# 4. MERGE WITH POKEMON DATA
data = pokemon_df.merge(win_percentage[['#', 'Win_Rate']], on='#', how='left')
data['Win_Rate'].fillna(0, inplace=True)

print(f'\nFinal dataset shape: {data.shape}')

# 5. EXPLORATORY DATA ANALYSIS
print('\n=== EXPLORATORY DATA ANALYSIS ===')
# Select numeric columns for analysis
numeric_cols = ['HP', 'Attack', 'Sp. Atk', 'Defense', 'Sp. Def', 'Speed', 'Win_Rate']
data_numeric = data[numeric_cols]

print(f'Numeric columns statistics:')
print(data_numeric.describe())

# Correlation matrix
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
corr_matrix = data_numeric.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, square=True)
plt.title('Correlation Matrix - Pokemon Stats')
plt.tight_layout()
plt.show()

print(f'\nTop correlations with Win_Rate:')
print(corr_matrix['Win_Rate'].sort_values(ascending=False))

# 6. FEATURE ENGINEERING
print('\n=== FEATURE ENGINEERING ===')
# Create derived features
data['Total_Stats'] = data['HP'] + data['Attack'] + data['Defense'] + data['Sp. Atk'] + data['Sp. Def'] + data['Speed']
data['Offensive_Power'] = data['Attack'] + data['Sp. Atk']
data['Defensive_Power'] = data['Defense'] + data['Sp. Def']
data['Speed_Ratio'] = data['Speed'] / (data['Speed'].max() + 1)

print(f'New features created: Total_Stats, Offensive_Power, Defensive_Power, Speed_Ratio')
print(f'Dataset shape after feature engineering: {data.shape}')

# 7. PREPARE DATA FOR MACHINE LEARNING
print('\n=== MACHINE LEARNING ===')
# Select features for modeling
feature_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Total_Stats', 'Offensive_Power', 'Defensive_Power', 'Speed_Ratio']
X = data[feature_cols].copy()
y = data['Win_Rate'].copy()

# Remove rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

# 8. TRAIN REGRESSION MODELS
print('\n=== TRAINING MODELS ===')
models = {
    'Linear Regression': LinearRegression(),
    'Support Vector Regression': SVR(kernel='rbf', C=100, gamma='scale'),
    'Decision Tree Regressor': DecisionTreeRegressor(max_depth=10, random_state=42)
}

results = {}
for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Evaluate
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_test = r2_score(y_test, y_pred_test)

    results[name] = {
        'MAE_Train': mae_train,
        'MAE_Test': mae_test,
        'RMSE_Test': rmse_test,
        'R2_Test': r2_test
    }

    print(f'{name}:')
    print(f'  MAE (Train): {mae_train:.4f}')
    print(f'  MAE (Test): {mae_test:.4f}')
    print(f'  RMSE (Test): {rmse_test:.4f}')
    print(f'  R2 (Test): {r2_test:.4f}\n')

# 9. MODEL COMPARISON
print('=== MODEL COMPARISON ===')
results_df = pd.DataFrame(results).T
print(results_df)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_df[['MAE_Train', 'MAE_Test']].plot(kind='bar', ax=axes[0])
axes[0].set_title('MAE Comparison - Train vs Test')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

results_df['R2_Test'].plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('R2 Score on Test Set')
axes[1].set_ylabel('R2 Score')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 10. DIMENSIONALITY REDUCTION WITH PCA
print('\n=== DIMENSIONALITY REDUCTION WITH PCA ===')
pca = PCA()
pca.fit(X_scaled)

# Calculate explained variance
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

print(f'Explained variance ratio: {explained_variance_ratio}')
print(f'Cumulative explained variance ratio: {cumulative_variance_ratio}')
print(f'Number of components for 95% variance: {np.argmax(cumulative_variance_ratio >= 0.95) + 1}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, 'bo-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance by Component')

axes[1].plot(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, 'ro-')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% variance')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.tight_layout()
plt.show()

print('\n=== ANALYSIS COMPLETE ===')
print('Pokemon win prediction analysis completed successfully!')
print(f'Best performing model: {results_df["MAE_Test"].idxmin()} with MAE = {results_df["MAE_Test"].min():.4f}')}